# Issues 1–7 — full-data run

This notebook uses the accompanying `src/` and `experiments/` code. Open it **inside the extracted project folder**, select your Python kernel, set the two real dataset paths below, and run from top to bottom. Keep the project files (`src/`, `experiments/`, `requirements.txt`) alongside this notebook. This notebook alone cannot run the experiments.

**Tracker mapping:** Issue 1 → #22; Issue 2 → #23; Issue 3 → baseline issue #24 ; Issue 4 → #25; Issue 5 → #26; Issue 6 → #27; Issue 7 → #28.

The run uses seeds **42, 7, 123, 1, 2024, 13, 21, 99, 2025, 314**. Per-seed results and exact split manifests are preserved.


## 0. Environment and project location

Install missing dependencies in the selected environment with `python -m pip install -r requirements.txt`.

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display, Image
PROJECT_DIR = Path(os.environ.get("ISSUE_PROJECT_DIR", Path.cwd())).expanduser().resolve()
os.chdir(PROJECT_DIR)
assert (PROJECT_DIR / "src/experiments/workflow.py").exists(), "Open this notebook in the extracted project folder."
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
print("Python:", sys.executable)
print("Project:", PROJECT_DIR)
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
from src.utils.io import get_git_commit
print("Verified source commit:", get_git_commit())

## Dataset paths and run configuration




In [ ]:
# Edit these two paths to point to actual dataset CSV files.
SCADANET_PATH = Path(os.environ.get("SCADANET_CSV", "datasets/scada_dataset_V01.csv")).expanduser().resolve()
BATADAL_PATH = Path(os.environ.get("BATADAL_CSV", "datasets/BATADAL_dataset04.csv")).expanduser().resolve()

for name, path in (("SCADANet", SCADANET_PATH), ("BATADAL", BATADAL_PATH)):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"{name} real dataset CSV is missing or empty: {path}. "
            "Edit the path above before running any issue."
        )

CONFIG = {
    "scadanet_path": str(SCADANET_PATH), "batadal_path": str(BATADAL_PATH),
    "seeds": [42, 7, 123, 1, 2024, 13, 21, 99, 2025, 314],
    "epochs": 300, "batadal_epochs": 100, "host_epochs": 400,
    "threshold": "validation", "cpu_threads": 4,
    "explanation_samples": 150, "topology_examples": 5,
    "resume": True, "profile": "full_data",
}
from src.experiments.workflow import configure, run_issue
RESULTS = configure(CONFIG)
(PROJECT_DIR / "workflow_config.json").write_text(json.dumps(CONFIG, indent=2))
print("Using real dataset files:", SCADANET_PATH, BATADAL_PATH, sep="\n  ")
display(pd.DataFrame([CONFIG]).drop(columns=["scadanet_path", "batadal_path"]))


## Issue 1 — Reusable pipeline and historical references

Runs the four original reference experiments and records provenance. Historical temporal protocols remain explicitly labeled; use Issue 5 for strict temporal evidence.

In [ ]:
run_issue(1, CONFIG)

In [ ]:
display(pd.DataFrame([json.loads(p.read_text()) for p in (RESULTS/'references').glob('*.json')])[["dataset","split_protocol","f1","git_commit"]])

## Issue 2 — Split audit

Audits the full original attack taxonomy, source/destination diversity, numeric feature drift, all five BATADAL CV folds, strict SCADANet topology, and three purged BATADAL boundaries. Coverage is computed from your data, never hardcoded to 2/13 or 12/13.

In [ ]:
run_issue(2, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'split_audits/split_audit_summary.csv'))

## Issue 3 — Fair baselines B1–B3

Runs all six Track-B families plus the principal strict temporal models on both datasets for each seed. Exact splits are replayed; preprocessing fits only training rows. Hyperparameters and thresholds use validation only. GCN is the prespecified alternative GNN; do not choose it using test scores.

In [ ]:
run_issue(3, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'baselines/table_b1_main_comparison.csv'))
display(pd.read_csv(RESULTS/'baselines/table_b2_temporal_comparison.csv'))

## Issue 4 — Topology/content ablations and interventions

Runs paired A–F conditions and feature interventions for every seed. Tables include uncertainty and deltas. Temporal D/E remain fixed-boundary rewiring controls; the principal topology comparison is Track B.

In [ ]:
run_issue(4, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'ablations/table_c1_trackb.csv').iloc[:, :12])
display(Image(filename=str(RESULTS/'ablations/figure_c1_trackb.png')))

## Issue 5 — Strict temporal protocols and matched recall

Runs T1/T2/T3 with one frozen model and threshold. Saves independent topology audits, per-window/per-attack results, matched Track-B recalls and deltas. BATADAL uses the same purge and threshold rules at all three boundaries. T2 uses a matched reference model, not a silently mismatched tuned B1 row.

In [ ]:
run_issue(5, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'temporal/table_t2_scadanet_per_attack.csv').iloc[:, :10])
display(pd.read_csv(RESULTS/'temporal/table_t3_batadal_split_sensitivity.csv').iloc[:, :10])
display(Image(filename=str(RESULTS/'temporal/figure_t4_static_vs_strict_heatmap.png')))

## Issue 6 — Host normalization H1–H4

Runs all four conditions under Track B and strict temporal for every seed. Separates threshold calibration from feature normalization. H3 is rebuilt from saved signed-log histograms. Retain attack-specific recall losses, host differences and temporal FPR increases.

In [ ]:
run_issue(6, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'host_normalization/table_h1_mitigation_comparison.csv').iloc[:, :10])
display(Image(filename=str(RESULTS/'host_normalization/figure_h2_precision_fpr_tradeoff.png')))
display(Image(filename=str(RESULTS/'host_normalization/figure_h3_distribution_frame_len.png')))

## Issue 7 — Robustness, operational metrics and explanations

Aggregates saved principal experiments, calculates paired statistics, and runs operational diagnostics plus stratified explanations/interventions for each seed. Reports cross-seed ranking stability. Time-based metrics remain N/A if exposure/timestamps or verified event labels do not support them.

In [ ]:
run_issue(7, CONFIG)

In [ ]:
display(pd.read_csv(RESULTS/'robustness/table_s2_statistical_comparison.csv'))
display(pd.read_csv(RESULTS/'robustness/table_s4_explainability_stability.csv'))

## Final consistency check and evidence summary

This checks paired seed coverage, model/protocol counts, recall deltas, causal topology, BATADAL zero overlap, saved provenance, and table/figure availability. Passing it confirms artifact consistency; scientific interpretation still requires the full-data results and their uncertainty.

In [ ]:
from experiments.validate_results import validate_results, write_interpretation
report = validate_results(CONFIG["seeds"])
print(f"Passed {report['n_checks']} artifact checks; {report['n_tables']} tables and {report['n_figures']} figures.")
print("Evidence summary:", write_interpretation())

## Save all results

The archive contains per-seed artifacts, split manifests, configurations, audits, tables and figures. 

In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT_DIR / "Issues_1_to_7_Results"), "zip", root_dir=PROJECT_DIR/'experiments', base_dir='results')
print(archive)
